# TEG Analysis — Quickstart

Run from the **repo root** (`teg_v2/`), or ensure the repo root is on your Python path.

All analytical functions live in `teg_analysis/`. This notebook loads the core data and demonstrates common starting points.

In [52]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

# Ensure repo root is on the path (only needed if not running from repo root)
repo_root = Path.cwd().parent if Path.cwd().name == 'ad_hoc_analysis' else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f'Repo root: {repo_root}')

Repo root: /Users/jon/projects/teg/teg_v2


In [53]:
# Core data loader
from teg_analysis.core.data_loader import load_all_data

# Analysis functions
from teg_analysis.analysis import (
    aggregate_data,
    get_teg_winners,
    get_best,
    get_worst,
)

# Constants
from teg_analysis.constants import PLAYER_DICT, ALL_DATA_PARQUET

print('Imports OK')
print(f'Players: {list(PLAYER_DICT.values())}')

Imports OK
Players: ['Alex BAKER', 'Jon BAKER', 'David MULLIN', 'Gregg WILLIAMS', 'Henry MELLER', 'Stuart NEUMANN', 'John PATTERSON', 'Graham PATTERSON']


In [54]:
# Load the main dataset (hole-level data, ~53 columns)
df = load_all_data()

print(f'Shape: {df.shape}')
print(f'TEGs: {sorted(df.TEGNum.unique())}')
print(f'Players: {sorted(df.Player.unique())}')
df.head()

Shape: (6390, 54)
TEGs: [np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18)]
Players: ['Alex BAKER', 'David MULLIN', 'Gregg WILLIAMS', 'Henry MELLER', 'John PATTERSON', 'Jon BAKER', 'Stuart NEUMANN']


,TEG,Round,Hole,PAR,SI,Pl,Sc,HC,HCStrokes,GrossVP,...,Stableford Round Avg,Stableford TEG Avg,Stableford Career Avg,TEG_Hole,Rank_GrossVP_TEG,Rank_Stableford_TEG,Gap_GrossVP_TEG,Gap_Stableford_TEG,Year,Area
0,TEG 7,1,1,5,7,AB,8.0,36.0,2.0,3.0,...,1.000000,1.000000,1.000000,1,6.0,5.0,4.0,3.0,2014,"Lisbon Coast, Portugal"
1,TEG 7,1,2,3,13,AB,4.0,36.0,2.0,1.0,...,2.000000,2.000000,2.000000,2,6.0,4.0,4.0,3.0,2014,"Lisbon Coast, Portugal"
2,TEG 7,1,3,5,11,AB,6.0,36.0,2.0,1.0,...,2.333333,2.333333,2.333333,3,5.0,4.0,5.0,3.0,2014,"Lisbon Coast, Portugal"
3,TEG 7,1,4,4,15,AB,5.0,36.0,2.0,1.0,...,2.500000,2.500000,2.500000,4,4.0,2.0,4.0,1.0,2014,"Lisbon Coast, Portugal"
4,TEG 7,1,5,3,17,AB,8.0,36.0,2.0,5.0,...,2.000000,2.000000,2.000000,5,6.0,5.0,9.0,4.0,2014,"Lisbon Coast, Portugal"


# Tanking Analysis

Investigates whether players score worse in the back half of tournaments (R3–R4) when out of contention, potentially to inflate their handicap.

**Primary metric:** GrossVP (gross vs par) — directly linked to handicap calculation.

---

## Part 1: Early vs Late Round Comparison

For each player × TEG: compare R1+R2 vs R3+R4 performance, both in absolute terms and relative to the field (to control for course difficulty).

In [55]:
# Aggregate to round level and filter to 4-round TEGs
round_df = aggregate_data(df, aggregation_level='Round')

rounds_per_teg = round_df.groupby('TEGNum')['Round'].nunique()
four_round_tegs = rounds_per_teg[rounds_per_teg == 4].index
r4_df = round_df[round_df['TEGNum'].isin(four_round_tegs)].copy()

print(f'Round-level rows: {len(round_df)}')
print(f'4-round TEGs: {sorted(four_round_tegs.tolist())}')
print(f'Excluded TEGs (not 4 rounds): {sorted(rounds_per_teg[rounds_per_teg != 4].index.tolist())}')

Round-level rows: 355
4-round TEGs: [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]
Excluded TEGs (not 4 rounds): [2]


In [56]:
# Part 1a: Absolute early/late delta per player per TEG
pivot = r4_df.pivot_table(
    index=['Player', 'TEGNum', 'TEG'],
    columns='Round',
    values='GrossVP',
    aggfunc='sum'
)
pivot.columns = [f'R{r}' for r in pivot.columns]
pivot = pivot.reset_index()

pivot['early_GrossVP'] = pivot['R1'] + pivot['R2']
pivot['late_GrossVP']  = pivot['R3'] + pivot['R4']
pivot['delta_abs']     = pivot['late_GrossVP'] - pivot['early_GrossVP']   # positive = worse in late rounds
pivot['r123_avg']      = (pivot['R1'] + pivot['R2'] + pivot['R3']) / 3
pivot['delta_r4']      = pivot['R4'] - pivot['r123_avg']                  # positive = R4 much worse than earlier avg

# Part 1b: Relative vs field (subtracts field mean per round per TEG)
field_means = r4_df.groupby(['TEGNum', 'Round'])['GrossVP'].mean().reset_index()
field_means = field_means.rename(columns={'GrossVP': 'field_mean'})
r4_rel = r4_df.merge(field_means, on=['TEGNum', 'Round'])
r4_rel['rel_GrossVP'] = r4_rel['GrossVP'] - r4_rel['field_mean']

pivot_rel = r4_rel.pivot_table(
    index=['Player', 'TEGNum', 'TEG'],
    columns='Round',
    values='rel_GrossVP',
    aggfunc='sum'
)
pivot_rel.columns = [f'R{r}_rel' for r in pivot_rel.columns]
pivot_rel = pivot_rel.reset_index()

result = pivot.merge(pivot_rel[['Player', 'TEGNum', 'TEG', 'R1_rel', 'R2_rel', 'R3_rel', 'R4_rel']],
                     on=['Player', 'TEGNum', 'TEG'])
result['early_rel']  = (result['R1_rel'] + result['R2_rel']) / 2
result['late_rel']   = (result['R3_rel'] + result['R4_rel']) / 2
result['delta_rel']  = result['late_rel'] - result['early_rel']   # positive = worse vs field in late rounds

print(f'{len(result)} player-TEG rows across {result.TEGNum.nunique()} TEGs')
result.head()

85 player-TEG rows across 16 TEGs


,Player,TEGNum,TEG,R1,R2,R3,R4,early_GrossVP,late_GrossVP,delta_abs,r123_avg,delta_r4,R1_rel,R2_rel,R3_rel,R4_rel,early_rel,late_rel,delta_rel
0,Alex BAKER,7,TEG 7,37.0,30.0,36.0,36.0,67.0,72.0,5.0,34.333333,1.666667,16.833333,12.666667,9.500000,16.000000,14.750000,12.750000,-2.000000
1,Alex BAKER,8,TEG 8,41.0,34.0,40.0,33.0,75.0,73.0,-2.0,38.333333,-5.333333,11.000000,9.333333,9.000000,6.166667,10.166667,7.583333,-2.583333
2,Alex BAKER,9,TEG 9,43.0,27.0,30.0,27.0,70.0,57.0,-13.0,33.333333,-6.333333,13.000000,4.666667,4.333333,5.166667,8.833333,4.750000,-4.083333
3,Alex BAKER,10,TEG 10,28.0,23.0,22.0,17.0,51.0,39.0,-12.0,24.333333,-7.333333,4.333333,3.666667,-2.166667,-5.333333,4.000000,-3.750000,-7.750000
4,Alex BAKER,11,TEG 11,28.0,20.0,24.0,34.0,48.0,58.0,10.0,24.000000,10.000000,1.000000,0.800000,1.200000,8.800000,0.900000,5.000000,4.100000


In [57]:
# Worst tanking instances: sorted by delta_abs (largest late-round deterioration first)
display_cols = ['Player', 'TEG', 'R1', 'R2', 'R3', 'R4',
                'early_GrossVP', 'late_GrossVP', 'delta_abs',
                'delta_r4', 'delta_rel']

tanking_instances = (
    result[display_cols]
    .sort_values('delta_abs', ascending=False)
    .reset_index(drop=True)
)
tanking_instances.round(1)

,Player,TEG,R1,R2,R3,R4,early_GrossVP,late_GrossVP,delta_abs,delta_r4,delta_rel
0,Henry MELLER,TEG 6,29.0,17.0,37.0,35.0,46.0,72.0,26.0,7.3,4.9
1,David MULLIN,TEG 5,12.0,11.0,27.0,22.0,23.0,49.0,26.0,5.3,10.4
2,Henry MELLER,TEG 7,12.0,10.0,24.0,23.0,22.0,47.0,25.0,7.7,8.0
3,Gregg WILLIAMS,TEG 6,20.0,32.0,35.0,41.0,52.0,76.0,24.0,12.0,3.9
4,Jon BAKER,TEG 7,11.0,14.0,26.0,20.0,25.0,46.0,21.0,3.0,6.0
...,...,...,...,...,...,...,...,...,...,...,...
80,Jon BAKER,TEG 15,27.0,29.0,13.0,23.0,56.0,36.0,-20.0,0.0,-5.9
81,David MULLIN,TEG 18,28.0,22.0,12.0,18.0,50.0,30.0,-20.0,-2.7,-7.3
82,Stuart NEUMANN,TEG 9,42.0,32.0,26.0,25.0,74.0,51.0,-23.0,-8.3,-9.1
83,John PATTERSON,TEG 13,36.0,33.0,12.0,25.0,69.0,37.0,-32.0,-2.0,-10.7


In [59]:
tanking_instances_rel = tanking_instances[['Player','TEG','delta_rel']]
tanking_instances_rel = tanking_instances_rel.sort_values('delta_rel', ascending=False)
tanking_instances_rel.round(1)

,Player,TEG,delta_rel
1,David MULLIN,TEG 5,10.4
10,Jon BAKER,TEG 18,9.7
5,David MULLIN,TEG 10,8.8
19,John PATTERSON,TEG 15,8.6
7,John PATTERSON,TEG 10,8.2
...,...,...,...
82,Stuart NEUMANN,TEG 9,-9.1
79,John PATTERSON,TEG 11,-10.4
76,Henry MELLER,TEG 5,-10.6
83,John PATTERSON,TEG 13,-10.7


In [ ]:
# Player summary: mean delta across all TEGs
# Positive delta_abs = player typically gets worse in later rounds
player_summary = (
    result.groupby('Player')
    .agg(
        n_tegs         =('delta_abs', 'count'),
        mean_delta_abs =('delta_abs', 'mean'),
        mean_delta_r4  =('delta_r4',  'mean'),
        mean_delta_rel =('delta_rel', 'mean'),
    )
    .sort_values('mean_delta_abs', ascending=False)
    .reset_index()
)
player_summary.round(1)

,Player,n_tegs,mean_delta_abs,mean_delta_r4,mean_delta_rel
0,Henry MELLER,6,7.8,2.7,1.4
1,David MULLIN,16,5.4,0.6,2.6
2,Jon BAKER,16,2.9,1.4,1.3
3,Gregg WILLIAMS,16,-0.1,0.4,-0.2
4,Alex BAKER,12,-2.9,-2.0,-0.9
5,John PATTERSON,8,-4.6,1.0,-1.4
6,Stuart NEUMANN,11,-5.9,-3.9,-3.9


---

## Part 2: Contention-Based Analysis

Does a player's R3+R4 performance worsen once they're out of contention after R2?

Two definitions of "out of contention":
- **Approach A (rank):** ranked in the bottom half of the field by cumulative GrossVP after R2
- **Approach B (gap):** more than a threshold number of strokes behind the leader after R2 (tested at 5 and 8 strokes)

In [ ]:
# Cumulative GrossVP after R2 for each player per TEG
r2_cum = (
    r4_df[r4_df['Round'] <= 2]
    .groupby(['TEGNum', 'TEG', 'Player'])['GrossVP']
    .sum()
    .reset_index()
    .rename(columns={'GrossVP': 'cum_r2_GrossVP'})
)

# Approach A: bottom half by rank
r2_cum['rank_after_r2']      = r2_cum.groupby('TEGNum')['cum_r2_GrossVP'].rank(method='min', ascending=True)
r2_cum['n_players']          = r2_cum.groupby('TEGNum')['Player'].transform('count')
r2_cum['in_contention_rank'] = r2_cum['rank_after_r2'] <= np.ceil(r2_cum['n_players'] / 2)

# Approach B: gap to leader (lower GrossVP = better gross = leading)
r2_cum['leader_r2']           = r2_cum.groupby('TEGNum')['cum_r2_GrossVP'].transform('min')
r2_cum['gap_to_leader']       = r2_cum['cum_r2_GrossVP'] - r2_cum['leader_r2']
r2_cum['in_contention_gap5']  = r2_cum['gap_to_leader'] <= 5
r2_cum['in_contention_gap8']  = r2_cum['gap_to_leader'] <= 8

# Merge in early/late/delta from Part 1 so comparisons are within-player
contention_df = r2_cum.merge(
    result[['Player', 'TEGNum', 'early_GrossVP', 'late_GrossVP', 'delta_abs', 'delta_rel']],
    on=['Player', 'TEGNum']
)

# Peek at the flags
contention_df[['Player', 'TEG', 'cum_r2_GrossVP', 'rank_after_r2', 'gap_to_leader',
               'in_contention_rank', 'in_contention_gap5', 'in_contention_gap8',
               'early_GrossVP', 'late_GrossVP', 'delta_abs']].round(1)


,Player,TEG,cum_r2_GrossVP,rank_after_r2,gap_to_leader,in_contention_rank,in_contention_gap5,in_contention_gap8,early_GrossVP,late_GrossVP,delta_abs
0,David MULLIN,TEG 3,38.0,1.0,0.0,True,True,True,38.0,34.0,-4.0
1,Gregg WILLIAMS,TEG 3,66.0,5.0,28.0,False,False,False,66.0,61.0,-5.0
2,Henry MELLER,TEG 3,46.0,4.0,8.0,False,False,True,46.0,50.0,4.0
3,Jon BAKER,TEG 3,45.0,3.0,7.0,True,False,True,45.0,49.0,4.0
4,Stuart NEUMANN,TEG 3,43.0,2.0,5.0,True,True,True,43.0,41.0,-2.0
...,...,...,...,...,...,...,...,...,...,...,...
80,Alex BAKER,TEG 18,60.0,5.0,21.0,False,False,False,60.0,64.0,4.0
81,David MULLIN,TEG 18,50.0,3.0,11.0,True,False,False,50.0,30.0,-20.0
82,Gregg WILLIAMS,TEG 18,41.0,2.0,2.0,True,True,True,41.0,25.0,-16.0
83,John PATTERSON,TEG 18,52.0,4.0,13.0,False,False,False,52.0,43.0,-9.0


In [ ]:
# Field-level comparison: mean delta (late − early GrossVP) for in vs out of contention
# delta_abs > 0 means the player scored worse in late rounds than early rounds

def contention_comparison(df, flag_col, label):
    summary = (
        df.groupby(flag_col)[['delta_abs', 'delta_rel']]
        .agg(n=('delta_abs', 'count'), mean_delta_abs=('delta_abs', 'mean'), mean_delta_rel=('delta_rel', 'mean'))
        .rename(index={True: 'In contention', False: 'Out of contention'})
        .round(1)
    )
    summary.index.name = label
    return summary

print('=== Approach A: bottom-half rank ===')
display(contention_comparison(contention_df, 'in_contention_rank', 'Rank approach'))

print('\n=== Approach B: >5 strokes behind leader ===')
display(contention_comparison(contention_df, 'in_contention_gap5', 'Gap ≤5 approach'))

print('\n=== Approach B: >8 strokes behind leader ===')
display(contention_comparison(contention_df, 'in_contention_gap8', 'Gap ≤8 approach'))


=== Approach A: bottom-half rank ===


,n,mean_delta_abs,mean_delta_rel
Rank approach,,,
Out of contention,38,-5.2,-2.9
In contention,47,5.1,2.3



=== Approach B: >5 strokes behind leader ===


,n,mean_delta_abs,mean_delta_rel
Gap ≤5 approach,,,
Out of contention,57,-2.6,-1.3
In contention,28,6.8,2.6



=== Approach B: >8 strokes behind leader ===


,n,mean_delta_abs,mean_delta_rel
Gap ≤8 approach,,,
Out of contention,50,-3.4,-1.8
In contention,35,6.1,2.5


In [ ]:
# Per-player breakdown: mean delta_abs when in vs out of contention (rank approach)
# Positive delta (out − in) means the player deteriorates more when out of contention
player_contention = (
    contention_df.groupby(['Player', 'in_contention_rank'])['delta_abs']
    .mean()
    .unstack('in_contention_rank')
    .rename(columns={True: 'In contention', False: 'Out of contention'})
    .round(1)
)
player_contention['extra deterioration (out − in)'] = (
    player_contention['Out of contention'] - player_contention['In contention']
).round(1)
player_contention.sort_values('extra deterioration (out − in)', ascending=False)


in_contention_rank,Out of contention,In contention,extra deterioration (out − in)
Player,,,
Gregg WILLIAMS,-0.6,0.4,-1.0
Stuart NEUMANN,-8.4,0.7,-9.1
Alex BAKER,-4.1,10.0,-14.1
John PATTERSON,-8.3,6.5,-14.8
Jon BAKER,-12.5,5.1,-17.6
Henry MELLER,-2.3,18.0,-20.3
David MULLIN,NaN,5.4,NaN
